# Feature Engineering - Master Dataset

Cilj: spojiti sve CSV fajlove u jedan master dataset i kreirati features za modeliranje.

Split:
- Train: 2013-01-01 do 2017-07-31
- Validation: 2017-08-01 do 2017-08-15 (zadnjih 15 dana, isti horizont kao Kaggle test)

Output:
- `data/processed/train_features.parquet`
- `data/processed/val_features.parquet`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import os

warnings.filterwarnings('ignore')

DATA      = '../data/raw/'
PROCESSED = '../data/processed/'
os.makedirs(PROCESSED, exist_ok=True)

EARTHQUAKE_DATE = pd.Timestamp('2016-04-16')
VAL_START       = pd.Timestamp('2017-08-01')

## 1. Ucitavanje podataka

In [ ]:
train        = pd.read_csv(DATA + 'train.csv', parse_dates=['date'])
stores       = pd.read_csv(DATA + 'stores.csv')
transactions = pd.read_csv(DATA + 'transactions.csv', parse_dates=['date'])
oil          = pd.read_csv(DATA + 'oil.csv', parse_dates=['date'])
holidays     = pd.read_csv(DATA + 'holidays_events.csv', parse_dates=['date'])

print(f'train: {train.shape}')
print(f'Period: {train.date.min().date()} do {train.date.max().date()}')
print(f'Val period: {VAL_START.date()} do {train.date.max().date()}')

## 2. Priprema pomocnih tabela

### 2a. Oil - interpolacija za vikende i praznike

In [ ]:
oil_full = (
    oil
    .set_index('date')
    .reindex(pd.date_range(train['date'].min(), train['date'].max()))
    .interpolate(method='linear')
    .reset_index()
    .rename(columns={'index': 'date', 'dcoilwtico': 'oil_price'})
)

print('Oil missing after interpolation:', oil_full['oil_price'].isnull().sum())

### 2b. Holidays - national i local flagovi

- `transferred=True` -> normalan dan, ignorisati
- `type='Bridge'` i `type='Work Day'` -> ignorisati
- National praznici vaze za sve prodavnice
- Local/Regional praznici vaze samo za prodavnice u tom gradu/regionu

In [ ]:
real_hol = holidays[
    (holidays['transferred'] == False) &
    (~holidays['type'].isin(['Bridge', 'Work Day']))
].copy()

national_hol = (
    real_hol[real_hol['locale'] == 'National']
    [['date']]
    .drop_duplicates()
    .assign(is_national_holiday=1)
)

local_hol = (
    real_hol[real_hol['locale'].isin(['Local', 'Regional'])]
    [['date', 'locale_name']]
    .drop_duplicates()
    .assign(is_local_holiday=1)
    .rename(columns={'locale_name': 'city'})
)

print('National holiday dates:', len(national_hol))
print('Local holiday records:', len(local_hol))

## 3. Merge - master dataset

In [ ]:
df = train.copy()

df = df.merge(stores,       on='store_nbr',          how='left')
df = df.merge(transactions, on=['date', 'store_nbr'], how='left')
df = df.merge(oil_full,     on='date',                how='left')
df = df.merge(national_hol, on='date',                how='left')
df = df.merge(local_hol,    on=['date', 'city'],      how='left')

df['is_national_holiday'] = df['is_national_holiday'].fillna(0).astype(int)
df['is_local_holiday']    = df['is_local_holiday'].fillna(0).astype(int)

df = df.sort_values(['store_nbr', 'family', 'date']).reset_index(drop=True)

print(f'Master shape: {df.shape}')
print(f'Kolone: {list(df.columns)}')

## 4. Kalendarske features

In [ ]:
df['dayofweek']  = df['date'].dt.dayofweek
df['month']      = df['date'].dt.month
df['weekofyear'] = df['date'].dt.isocalendar().week.astype(int)
df['dayofmonth'] = df['date'].dt.day
df['year']       = df['date'].dt.year
df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)

df['days_in_month'] = df['date'].dt.days_in_month
df['is_payday'] = (
    (df['dayofmonth'] == 15) | (df['dayofmonth'] == df['days_in_month'])
).astype(int)
df.drop(columns=['days_in_month'], inplace=True)

print('Kalendarske features dodane.')

## 5. Earthquake feature

In [ ]:
days_after = (df['date'] - EARTHQUAKE_DATE).dt.days
df['days_after_earthquake'] = np.where(
    days_after < 0, 0, days_after.clip(upper=30)
)

print('Earthquake feature dodan.')

## 6. Target transformacija i lag/rolling features

Lagi se racunaju po grupi (store_nbr, family).
Koristimo shift(1) kao polaziste za rolling - da ne curcamo buducu informaciju u proslost.

In [ ]:
df['sales_log'] = np.log1p(df['sales'])

LAG_DAYS     = [7, 14, 28, 56]
ROLL_WINDOWS = [7, 14, 28]

def add_lags_and_rolling(group):
    group = group.sort_values('date')
    for lag in LAG_DAYS:
        group[f'lag_{lag}'] = group['sales_log'].shift(lag)
    for w in ROLL_WINDOWS:
        group[f'roll_mean_{w}'] = group['sales_log'].shift(1).rolling(w).mean()
        group[f'roll_std_{w}']  = group['sales_log'].shift(1).rolling(w).std()
    return group

print('Racunam lag i rolling features (moze potrajati par minuta)...')
df = df.groupby(['store_nbr', 'family'], group_keys=False).apply(add_lags_and_rolling)
print('Gotovo.')

lag_cols = [f'lag_{l}' for l in LAG_DAYS] + \
           [f'roll_mean_{w}' for w in ROLL_WINDOWS] + \
           [f'roll_std_{w}'  for w in ROLL_WINDOWS]

## 7. Encoding kategorijskih varijabli

In [ ]:
df['family_enc'] = df['family'].astype('category').cat.codes
df['type_enc']   = df['type'].astype('category').cat.codes
df['city_enc']   = df['city'].astype('category').cat.codes
df['state_enc']  = df['state'].astype('category').cat.codes

family_map = dict(enumerate(df['family'].astype('category').cat.categories))
type_map   = dict(enumerate(df['type'].astype('category').cat.categories))

print('Encoding gotov.')

## 8. Train / Validation split

In [ ]:
feature_cols = [
    'date', 'store_nbr', 'family',
    'sales', 'sales_log',
    'city', 'state', 'type', 'cluster',
    'family_enc', 'type_enc', 'city_enc', 'state_enc',
    'onpromotion', 'transactions', 'oil_price',
    'year', 'month', 'weekofyear', 'dayofweek', 'dayofmonth',
    'is_weekend', 'is_payday',
    'is_national_holiday', 'is_local_holiday',
    'days_after_earthquake',
] + lag_cols

final = df[feature_cols].copy()

# Izbaci redove gdje lag_56 nije dostupan (prvih ~56 dana po grupi)
final = final.dropna(subset=['lag_56'])

train_df = final[final['date'] <  VAL_START].copy()
val_df   = final[final['date'] >= VAL_START].copy()

print(f'Train: {train_df.shape}  ({train_df.date.min().date()} do {train_df.date.max().date()})')
print(f'Val:   {val_df.shape}  ({val_df.date.min().date()} do {val_df.date.max().date()})')

In [ ]:
# Distribucija sales_log
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(train_df['sales'].clip(upper=2000), bins=80, color='steelblue', edgecolor='none')
axes[0].set_title('Distribucija sales (clipped na 2000)')
axes[0].set_xlabel('sales')

axes[1].hist(train_df['sales_log'], bins=80, color='darkorange', edgecolor='none')
axes[1].set_title('Distribucija log1p(sales)')
axes[1].set_xlabel('log1p(sales)')

plt.tight_layout()
plt.show()

In [ ]:
# Korelacija features sa targetom
num_cols = [
    'sales_log', 'onpromotion', 'oil_price', 'transactions',
    'is_national_holiday', 'is_local_holiday', 'is_payday', 'is_weekend',
    'days_after_earthquake', 'cluster',
    'lag_7', 'lag_14', 'lag_28', 'roll_mean_7', 'roll_mean_28'
]

corr = train_df[num_cols].corr()['sales_log'].drop('sales_log').sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['salmon' if x < 0 else 'steelblue' for x in corr]
corr.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Korelacija features sa log1p(sales)')
ax.set_xlabel('Pearson r')
plt.tight_layout()
plt.show()

print(corr.to_string())

## 9. Snimanje na disk

In [ ]:
train_df.to_parquet(PROCESSED + 'train_features.parquet', index=False)
val_df.to_parquet(PROCESSED   + 'val_features.parquet',   index=False)

import json
with open(PROCESSED + 'encodings.json', 'w') as f:
    json.dump({'family': family_map, 'type': type_map}, f, indent=2)

print('Snimljeno:')
print(f'  {PROCESSED}train_features.parquet  ({train_df.shape[0]:,} redova)')
print(f'  {PROCESSED}val_features.parquet    ({val_df.shape[0]:,} redova)')
print(f'  {PROCESSED}encodings.json')